## Imports

In [ ]:
import numpy as np

import torch
import torch.nn as nn
from torchsummary import summary
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
import torch.nn.functional as F
from torch_geometric.nn import SAGEConv
from torch.utils.tensorboard import SummaryWriter

import matplotlib.pyplot as plt
import pickle
from datetime import datetime
import time

import optuna

## LOADING THE DATA

In [ ]:
path = '../data/final_dataset/graph/train_dataset/graphs_delaunay_norotation_local.pt'
#path = '../data/final_dataset/graph/train_dataset/graphs_sequential_norotation_local.pt'
dataset = torch.load(path, weights_only=False)

In [ ]:
# Split ratios
train_ratio = 0.5
val_ratio = 0.35 
test_ratio = 0.15
total = len(dataset)
model_type = ''

batch_size = 1 

dataset = [data.sort(sort_by_row=False) for data in dataset]

train_dataset = dataset[:int(total * train_ratio)]
val_dataset   = dataset[int(total * train_ratio):int(total * (train_ratio + val_ratio))]
test_dataset  = dataset[int(total * (train_ratio + val_ratio)):]
print("Train Dataset Size: ", len(train_dataset))
print("Val Dataset Size: ", len(val_dataset))
print("Test Dataset Size: ", len(test_dataset))

if 'sequential' in path:
    model_type = 'seq'
    sorted_indices = sorted(range(len(train_dataset)))
    train_dataset = [train_dataset[i] for i in sorted_indices]
    val_dataset   = [val_dataset[i] for i in sorted(range(len(val_dataset)))]
    test_dataset  = [test_dataset[i] for i in sorted(range(len(test_dataset)))]
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=False) 
    print("Sequential dataset being used!")

else:
    model_type = 'del'
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True) 
    print("Delaunay dataset being used!")

val_loader   = DataLoader(val_dataset, batch_size=batch_size, shuffle=False) 
test_loader  = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [ ]:
# SAVING TEST DATASET
# with open(f'../data/final_dataset/graph/graph_{model_type}_test_data_newsyn.pkl', 'wb') as f:
#     pickle.dump(test_dataset, f)

## TRAINING THE MODEL

In [ ]:
# DEVICE
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

In [ ]:
# MODEL ARCHITECTURE GraphSage with DELAUNAY

class GraphSAGEDelaunay(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, dropout=0.3):
        super().__init__()
        self.dropout = dropout

        self.conv1 = SAGEConv(in_channels, hidden_channels)
        self.bn1 = nn.LayerNorm(hidden_channels)
        self.conv2 = SAGEConv(hidden_channels, out_channels)

    def forward(self, x, edge_index):
        fixed = (x[:, 0:1] == 0).float()   
        mask = 1.0 - fixed

        h = self.conv1(x, edge_index)
        h = self.bn1(h)
        h = F.relu(h)
        h = F.dropout(h, p=self.dropout, training=self.training)
        out = self.conv2(h, edge_index)

        return out * mask


In [ ]:
# MODEL ARCHITECTURE GraphSage with LSTM and SEQUENTIAL ORDERING

class GraphSAGE(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, dropout=0.3):
        super().__init__()
        self.dropout = dropout

        self.conv1 = SAGEConv(in_channels, hidden_channels, 'lstm')
        self.bn1 = nn.LayerNorm(hidden_channels)
        self.conv2 = SAGEConv(hidden_channels, out_channels)

    def forward(self, x, edge_index):
        fixed = (x[:, 0:1] == 0).float()   
        mask = 1.0 - fixed                

        h = self.conv1(x, edge_index)
        h = self.bn1(h)
        h = F.relu(h)
        h = F.dropout(h, p=self.dropout, training=self.training)
        out = self.conv2(h, edge_index)

        return out * mask


In [ ]:
# MODEL PARAMETERS GraphSage 

sample = dataset[0]
in_channels = sample.num_features
out_channels = sample.y.shape[1]

# Model parameter
dropout = 0.4 
learning_rate = 1e-3
weight_decay = 1e-5

# Creating the model 
if 'delaunay' in path:
    hidden_channels = 128 
    learning_rate = 3.692e-4
    model = GraphSAGEDelaunay(in_channels, hidden_channels, out_channels, dropout).to(device)
    description = 'newData_Delaunay_Huber_Combined' 
    print('Delaunay model selected!')

else:
    hidden_channels = 64
    learning_rate = 2.807e-4
    model = GraphSAGE(in_channels, hidden_channels, out_channels, dropout).to(device)
    description = 'newData_Sequential_Huber_Combined'
    print('Sequential model selected!')

# Optimizer
optimizer = torch.optim.RMSprop(model.parameters(), lr=learning_rate, weight_decay=weight_decay)

# Loss Function
criterion_huber = nn.SmoothL1Loss()
criterion_mse = nn.MSELoss()

# Scheduler
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=5
)

In [ ]:
# EARLY STOPPING CALLBACK 

class EarlyStopping:
    def __init__(self, patience=15, delta=0):
        self.patience = patience
        self.delta = delta
        self.best_loss = float('inf')
        self.counter = 0
        self.early_stop = False

    def __call__(self, val_loss):
        if val_loss < self.best_loss - self.delta:
            self.best_loss = val_loss
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True

early_stopping = EarlyStopping(patience=30, delta=1e-5)

In [ ]:
#SAVES MODEL STATE

def save_checkpoint(epoch, model, optimizer, train_losses, val_losses, path=''):
    checkpoint = {
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "train_losses": train_losses,
        "val_losses": val_losses,
    }
    torch.save(checkpoint, path)
    print(f"Checkpoint saved to {path}")


In [ ]:
#LOADS STATE TO RESUME TRAINING

def load_checkpoint(path, model, optimizer=None):
    checkpoint = torch.load(path, map_location="cpu")
    
    model.load_state_dict(checkpoint["model_state_dict"])
    
    if optimizer is not None:
        optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
    
    train_losses = checkpoint.get("train_losses", [])
    val_losses = checkpoint.get("val_losses", [])
    start_epoch = checkpoint["epoch"] + 1

    print(f"Loaded checkpoint from epoch {checkpoint['epoch']}")
    
    return start_epoch, train_losses, val_losses

In [ ]:
lambda_disp = 2e-1

def train_epoch(loader):
    model.train()
    total_loss = 0
    for batch in loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        out = model(batch.x, batch.edge_index)
        mask = batch.x[:,0] == 1    

        #CUSTOM LOSS FUNCTION 
        main_loss = criterion_huber(out[mask], batch.y[mask])  
        disp_loss = torch.mean(torch.sum(out[mask]**2, dim=1))
        loss = main_loss + lambda_disp * disp_loss

        #loss = criterion_mse(out[mask], batch.y[mask])
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

@torch.no_grad()
def evaluate(loader):
    model.eval()
    total_loss = 0
    for batch in loader:
        batch = batch.to(device)
        out = model(batch.x, batch.edge_index)
        mask = batch.x[:,0] == 1   

        #CUSTOM LOSS FUNCTION    
        main_loss = criterion_huber(out[mask], batch.y[mask])
        disp_loss = torch.mean(torch.sum(out[mask]**2, dim=1))
        loss = main_loss + lambda_disp * disp_loss

        #loss = criterion_mse(out[mask], batch.y[mask])
        total_loss += loss.item()
    return total_loss / len(loader)


### TRAINING LOOP

In [ ]:
epochs = 300
train_losses, val_losses = [], []
best_val_loss = float("inf")
timestamp = datetime.now().strftime("%Y%m%d-%H%M")
epochs_completed = 0
log_dir = f'../checkpoints/graph/logs/{timestamp}'
writer = SummaryWriter(log_dir)
epoch_times = []

# Reload and resume training
#start_epoch, train_losses, val_losses = load_checkpoint("checkpoint.pt", model, optimizer)

for epoch in range(1, epochs + 1):
    start = time.time()
    epochs_completed +=1
    train_loss = train_epoch(train_loader)
    val_loss = evaluate(val_loader)

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    # Log values to TensorBoard
    writer.add_scalar("Loss/train", train_loss, epoch)
    writer.add_scalar("Loss/val", val_loss, epoch)
    writer.add_scalar("LearningRate", scheduler.optimizer.param_groups[0]['lr'], epoch)

    scheduler.step(val_loss)        # LR scheduling
    early_stopping(val_loss)       # early stopping

    print(f"Epoch {epoch:02d} | Train Loss: {train_loss:.6f} | Val Loss: {val_loss:.6f}")

    # Save every epoch:
    #save_checkpoint(epoch, model, optimizer, train_losses, val_losses, "checkpoint.pt")

    #saves only save when validation improves:
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        save_checkpoint(epoch, model, optimizer, train_losses, val_losses, f'../checkpoints/graph/model/{timestamp}_{description}_{batch_size}_batches_{epochs}_epochs.pt')

    epoch_times.append(time.time() - start)
    
    if early_stopping.early_stop:
        print("Early stopping triggered.")
        break

#writer.add_graph(model, x_sample)
writer.close()
total_time = sum(epoch_times)
print("Avg time per epoch:", sum(epoch_times) / len(epoch_times))
print("Total elapsed time:", total_time)

# SAVING LOSSES
np.save(f'../checkpoints/graph/history/{timestamp}_{description}_{batch_size}_batches_{epochs}_epochs_train_loss', train_losses)
np.save(f'../checkpoints/graph/history/{timestamp}_{description}_{batch_size}_batches_{epochs}_epochs_val_loss', val_losses)

In [ ]:
#sum(epoch_times)
print(model)

In [ ]:
for name, param in model.named_parameters():
    writer.add_histogram(name, param, epoch)
    if param.grad is not None:
        writer.add_histogram(f'{name}.grad', param.grad, epoch)


In [ ]:
# %reload_ext tensorboard
# %tensorboard --logdir ../checkpoints/graph/logs

## POST TRAINING

### PLOTTING TRAINING LOSS

In [ ]:
# Load the checkpoint
checkpoint = torch.load('../checkpoints/graph/model/Delaunay_Huber_Combined_1_batches_300_epochs.pt', map_location='cpu')

# Load only the model weights
model.load_state_dict(checkpoint['model_state_dict'])

In [ ]:
# PLOT TRAINING & VALIDATION LOSS 

plt.figure(figsize=(8,6))
plt.plot(range(epochs_completed), train_losses[:epochs_completed], label='Train Loss', color='#143642') 
plt.plot(range(epochs_completed), val_losses[:epochs_completed], label='Validation Loss', color='#EC9A29')
plt.xlabel('Epoch', fontdict=labels_font)
plt.ylabel('Loss', fontdict=labels_font)
plt.title(f'Training & Validation Loss over {epochs_completed} Epochs', fontdict=title_font)
plt.legend(prop=legend_font)
plt.grid(True)

plt.xticks(fontsize=tick_font['size'], family=tick_font['family'])
plt.yticks(fontsize=tick_font['size'], family=tick_font['family'])

#save figure
save_path = f'../checkpoints/graph/history/{timestamp}_{description}_{batch_size}_batches_{epochs}_epochs.png'
plt.savefig(save_path, dpi=300, bbox_inches='tight')

plt.show()

### PLOTTING PREDICTIONS

In [ ]:
# MAKE PREDICTIONS AND STORE THEM IN LIST
model.eval()

all_predictions = []  # list of dicts

with torch.no_grad():
    for data in test_dataset:
        data = data.to(device)

        pred_shift = model(data.x, data.edge_index).cpu().numpy()
        true_shift = data.y[:,0:2].cpu().numpy()
        coords = data.x[:,1:3].cpu().numpy()
        line_id = data.x[:,0].cpu().numpy()

        all_predictions.append({
            "coords": coords,
            "true_shift": true_shift,
            "pred_shift": pred_shift,
            "line_id": line_id
        })

In [ ]:
#SAVE RESULTS
with open(f'../data/results/graph_{model_type}/{timestamp}_{description}_{batch_size}_batches_{epochs}_epochs_predictions.pkl', 'wb') as f:
    pickle.dump(all_predictions, f)


### PUSH DATASET

In [ ]:
import torch
from torch_geometric.loader import DataLoader
import torch.nn.functional as F

push_dataset = []
for p in paths:
    d = torch.load(p, weights_only=False)
    d = d.sort(sort_by_row=False)
    push_dataset.append(d)

push_loader = DataLoader(push_dataset, batch_size=1, shuffle=True)

for p in model.conv1.parameters():
    p.requires_grad = True
for p in model.bn1.parameters():
    p.requires_grad = False

def forward_with_mask(model, data):
    out = model(data.x, data.edge_index)
    fixed = (data.x[:,0:1] == 0).float()
    mask = 1 - fixed
    return out * mask

optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
criterion = torch.nn.SmoothL1Loss()

model.train()
for epoch in range(10):
    total_loss = 0.0
    for data in push_loader:
        optimizer.zero_grad()
        pred = forward_with_mask(model, data)
        loss = criterion(pred, data.y[:,0:2])
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    if epoch % 50 == 0:
        print(f"Epoch {epoch:4d} | Push Loss: {total_loss/len(push_loader):.6f}")

model.eval()
data = push_dataset[0]
with torch.no_grad():
    pred = forward_with_mask(model, data).cpu().numpy()

coords = data.x[:,1:3].cpu().numpy()
true_shift = data.y[:,0:2].cpu().numpy()
line_id = data.x[:,0].cpu().numpy()

pred_push = [{
    "coords": coords,
    "true_shift": true_shift,
    "pred_shift": pred,
    "line_id": line_id
}]

In [ ]:
model.eval()

for i, data in enumerate(push_dataset):
    with torch.no_grad():
        pred = forward_with_mask(model, data).cpu().numpy()

    coords = data.x[:,1:3].cpu().numpy()
    true_shift = data.y[:,0:2].cpu().numpy()
    line_id = data.x[:,0].cpu().numpy()

    pred_push = [{
        "coords": coords,
        "true_shift": true_shift,
        "pred_shift": pred,
        "line_id": line_id
    }]

## MODEL OPTIMIZATION

In [ ]:
def train_and_evaluate(
    hidden_channels,
    optimizer_name,
    learning_rate,
    batch_size,
    trial=None
):
    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=('delaunay' in path)
    )
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

    # Model
    if 'delaunay' in path:
        model = GraphSAGEDelaunay(
            in_channels, hidden_channels, out_channels, dropout=0.0
        ).to(device)
    else:
        model = GraphSAGE(
            in_channels, hidden_channels, out_channels, dropout=0.0
        ).to(device)

    # Optimizer
    if optimizer_name == "Adam":
        optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    elif optimizer_name == "AdamW":
        optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
    elif optimizer_name == "RMSprop":
        optimizer = torch.optim.RMSprop(model.parameters(), lr=learning_rate)
    else:
        raise ValueError(f"Unknown optimizer {optimizer_name}")

    criterion = nn.SmoothL1Loss()
    best_val_loss = float("inf")

    # Unique filename per trial (important!)
    model_path = (
        f"best_model_trial_{trial.number}.pt"
        if trial is not None
        else "best_model.pt"
    )

    for epoch in range(100):
        # ---- Training ----
        model.train()
        for batch in train_loader:
            batch = batch.to(device)
            optimizer.zero_grad()
            out = model(batch.x, batch.edge_index)
            mask = batch.x[:, 0] == 1
            loss = criterion(out[mask], batch.y[mask])
            loss.backward()
            optimizer.step()

        # ---- Validation ----
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for batch in val_loader:
                batch = batch.to(device)
                out = model(batch.x, batch.edge_index)
                mask = batch.x[:, 0] == 1
                loss = criterion(out[mask], batch.y[mask])
                val_loss += loss.item()
        val_loss /= len(val_loader)

        # ---- Save best model ----
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), model_path)

    return best_val_loss

In [ ]:
def objective(trial):
    hidden_channels = trial.suggest_categorical("hidden_channels", [32, 64, 128])
    optimizer_name = trial.suggest_categorical("optimizer", ["Adam", "AdamW", "RMSprop"])
    learning_rate = trial.suggest_float("lr", 1e-5, 1e-2, log=True)
    batch_size = trial.suggest_categorical("batch_size", [1, 2, 4, 8])

    val_loss = train_and_evaluate_simple(
        hidden_channels=hidden_channels,
        optimizer_name=optimizer_name,
        learning_rate=learning_rate,
        batch_size=batch_size
    )

    return val_loss

In [ ]:
def print_status(study, trial):
    print(f"Trial {trial.number} finished.")
    print(f"  Validation Loss: {trial.value:.6f}")
    print(f"  Best Loss so far: {study.best_value:.6f}")
    print(f"  Best Hyperparameters so far: {study.best_params}\n")

In [ ]:
study = optuna.create_study(
    direction="minimize",
    study_name="GraphSAGE_BO_Del"
)

study.optimize(
    objective, 
    n_trials=20,
    show_progress_bar=True, 
    callbacks=[print_status], 
    timeout=None
)

In [ ]:
print("Best validation loss:", study.best_value)
print("Best hyperparameters:")
for k, v in study.best_params.items():
    print(f"  {k}: {v}")


In [ ]:
best = study.best_params

final_val_loss = train_and_evaluate_simple(
    hidden_channels=best["hidden_channels"],
    optimizer_name=best["optimizer"],
    learning_rate=best["lr"],
    batch_size=best["batch_size"],
)